# 📈 Linear Regression — Solutions Notebook

**This notebook contains complete, verified solutions.**  
Try the practice notebook first before looking at these!

---

## 🎯 Section 1: Overview

**Linear Regression** models the relationship between dependent variable (target) and independent variables (features) by fitting a linear equation.

- **Simple**: `y = θ₀ + θ₁x`
- **Multiple**: `y = θ₀ + θ₁x₁ + θ₂x₂ + ... + θₙxₙ`

### Assumptions (LINE)
1. **Linearity** — relationship between X and y is linear
2. **Independence** — observations are independent
3. **Normality** — residuals are normally distributed
4. **Equal variance** — residuals have constant variance

## 📐 Section 2: Math & Intuition

### Hypothesis: $h_\theta(x) = \theta^T x = X\theta$

### Cost Function (MSE):
$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2$$

### Gradient Descent Update:
$$\theta := \theta - \frac{\alpha}{m} X^T(X\theta - y)$$

### Normal Equation:
$$\theta = (X^T X)^{-1} X^T y$$

---
## 🔧 Section 3: Implementation from Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

### 3.1 Generate Synthetic Data

In [ ]:
# Generate synthetic data: y = 4 + 3x + noise
m = 100
X = 2 * np.random.rand(m, 1)
y = 4 + 3 * X + np.random.randn(m, 1) * 0.5

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.7, edgecolors='k', linewidth=0.5)
plt.xlabel('X (Feature)', fontsize=12)
plt.ylabel('y (Target)', fontsize=12)
plt.title('Synthetic Linear Data', fontsize=14)
plt.show()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'True parameters: θ₀ = 4, θ₁ = 3')

### 3.2 Add Bias Term

In [ ]:
# ✅ SOLUTION: Add bias column
X_b = np.c_[np.ones((m, 1)), X]

assert X_b.shape == (100, 2)
assert np.all(X_b[:, 0] == 1)
print(f'X_b shape: {X_b.shape} ✅')
print(f'First 3 rows:\n{X_b[:3]}')

### 3.3 Normal Equation

In [ ]:
# ✅ SOLUTION: Normal equation
def normal_equation(X, y):
    """
    Compute optimal theta using the normal equation.
    θ = (X^T X)^(-1) X^T y
    """
    theta = np.linalg.inv(X.T @ X) @ X.T @ y
    return theta


theta_ne = normal_equation(X_b, y)
print(f'Normal Equation Result:')
print(f'  θ₀ (intercept) = {theta_ne[0, 0]:.4f}  (expected ≈ 4.0)')
print(f'  θ₁ (slope)     = {theta_ne[1, 0]:.4f}  (expected ≈ 3.0)')

assert abs(theta_ne[0, 0] - 4.0) < 1.0
assert abs(theta_ne[1, 0] - 3.0) < 1.0
print('\n✅ Normal equation looks correct!')

### 3.4 Gradient Descent

In [ ]:
# ✅ SOLUTION: Cost function
def compute_cost(X, y, theta):
    """
    Compute MSE cost: J(θ) = (1/2m) * sum((Xθ - y)²)
    """
    m = len(y)
    predictions = X @ theta
    errors = predictions - y
    cost = (1 / (2 * m)) * np.sum(errors ** 2)
    return cost


theta_test = np.array([[0], [0]])
print(f'Cost with θ=[0,0]: {compute_cost(X_b, y, theta_test):.4f}')
print(f'Cost with θ from normal eq: {compute_cost(X_b, y, theta_ne):.4f}')

In [ ]:
# ✅ SOLUTION: Gradient descent
def gradient_descent(X, y, theta, alpha, num_iters):
    """
    Batch gradient descent for linear regression.
    """
    m = len(y)
    history = []
    
    for i in range(num_iters):
        predictions = X @ theta
        errors = predictions - y
        gradients = (1 / m) * X.T @ errors
        theta = theta - alpha * gradients
        history.append(compute_cost(X, y, theta))
    
    return theta, history


theta_init = np.zeros((2, 1))
alpha = 0.1
num_iters = 1000

theta_gd, cost_history = gradient_descent(X_b, y, theta_init.copy(), alpha, num_iters)

print(f'Gradient Descent Result:')
print(f'  θ₀ (intercept) = {theta_gd[0, 0]:.4f}  (expected ≈ 4.0)')
print(f'  θ₁ (slope)     = {theta_gd[1, 0]:.4f}  (expected ≈ 3.0)')
print(f'  Final cost      = {cost_history[-1]:.6f}')

# Verify gradient descent converges to same result as normal equation
assert abs(theta_gd[0, 0] - theta_ne[0, 0]) < 0.01, "GD should match normal equation"
assert abs(theta_gd[1, 0] - theta_ne[1, 0]) < 0.01, "GD should match normal equation"
print('\n✅ Gradient descent matches normal equation!')

### 3.5 Visualize Convergence

In [ ]:
# ✅ SOLUTION: Convergence plot
plt.figure(figsize=(10, 6))
plt.plot(cost_history, linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Cost J(θ)', fontsize=12)
plt.title('Gradient Descent Convergence', fontsize=14)
plt.yscale('log')  # log scale to see convergence better
plt.grid(True, alpha=0.3)
plt.show()

### 3.6 Visualize the Fit

In [ ]:
# ✅ SOLUTION: Plot data and regression line
plt.figure(figsize=(10, 6))

# Scatter plot
plt.scatter(X, y, alpha=0.7, edgecolors='k', linewidth=0.5, label='Data')

# Regression line
X_plot = np.linspace(0, 2, 100).reshape(-1, 1)
X_plot_b = np.c_[np.ones((100, 1)), X_plot]
y_plot = X_plot_b @ theta_gd

plt.plot(X_plot, y_plot, 'r-', linewidth=2, label=f'Fit: y = {theta_gd[0,0]:.2f} + {theta_gd[1,0]:.2f}x')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Linear Regression Fit', fontsize=14)
plt.legend(fontsize=11)
plt.show()

---
## 📦 Section 4: Using scikit-learn

In [ ]:
# ✅ SOLUTION: scikit-learn linear regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Step 1: Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Create and fit model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 3: Predict
y_pred = model.predict(X_test)

# Step 4: Print coefficients
print(f'Intercept (θ₀): {model.intercept_[0]:.4f}  (expected ≈ 4.0)')
print(f'Slope (θ₁):     {model.coef_[0, 0]:.4f}  (expected ≈ 3.0)')

# Step 5: Evaluate
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'\n--- Test Set Metrics ---')
print(f'MSE:  {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'R²:   {r2:.4f}')

### 4.1 Multiple Linear Regression

In [ ]:
# ✅ SOLUTION: Multiple linear regression
np.random.seed(42)
m_multi = 200
X_multi = np.random.randn(m_multi, 3)
y_multi = 2 + 3 * X_multi[:, 0] - 1.5 * X_multi[:, 1] + 0.5 * X_multi[:, 2] + np.random.randn(m_multi) * 0.3

model_multi = LinearRegression()
model_multi.fit(X_multi, y_multi)

print(f'Intercept: {model_multi.intercept_:.4f}  (expected ≈ 2.0)')
print(f'Coefficients: {model_multi.coef_}')
print(f'Expected:     [3.0, -1.5, 0.5]')

# Verify
r2_multi = model_multi.score(X_multi, y_multi)
print(f'\nR² on training data: {r2_multi:.4f} (should be very high ~0.99)')

---
## 🧪 Section 5: Experiments

### 5.1 Effect of Learning Rate

In [ ]:
# ✅ SOLUTION: Learning rate comparison
alphas = [0.005, 0.1, 0.5]
colors = ['#e74c3c', '#2ecc71', '#3498db']

plt.figure(figsize=(12, 6))

for alpha_val, color in zip(alphas, colors):
    theta_init = np.zeros((2, 1))
    _, history = gradient_descent(X_b, y, theta_init.copy(), alpha_val, 500)
    plt.plot(history, label=f'α = {alpha_val}', linewidth=2, color=color)

plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Cost J(θ)', fontsize=12)
plt.title('Effect of Learning Rate on Convergence', fontsize=14)
plt.legend(fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

print('Observations:')
print('• α=0.005: Too slow — needs many more iterations')
print('• α=0.1:   Just right — smooth, fast convergence')
print('• α=0.5:   Faster initially but may oscillate with larger values')

### 5.2 Feature Scaling Impact

In [ ]:
# ✅ SOLUTION: Feature scaling experiment
np.random.seed(42)
X_unscaled = np.column_stack([
    np.random.randn(100) * 1000,
    np.random.randn(100) * 0.001
])
y_unscaled = (5 + 2 * X_unscaled[:, 0] + 3000 * X_unscaled[:, 1] 
              + np.random.randn(100) * 10).reshape(-1, 1)

# WITHOUT scaling
X_b_unscaled = np.c_[np.ones((100, 1)), X_unscaled]
theta_init = np.zeros((3, 1))
# Use very small learning rate to avoid divergence
_, history_unscaled = gradient_descent(X_b_unscaled, y_unscaled, theta_init.copy(), 1e-7, 1000)

# WITH scaling
X_mean = X_unscaled.mean(axis=0)
X_std = X_unscaled.std(axis=0)
X_scaled = (X_unscaled - X_mean) / X_std
X_b_scaled = np.c_[np.ones((100, 1)), X_scaled]
theta_init = np.zeros((3, 1))
_, history_scaled = gradient_descent(X_b_scaled, y_unscaled, theta_init.copy(), 0.1, 1000)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_unscaled, 'r-', linewidth=2)
axes[0].set_title('WITHOUT Feature Scaling', fontsize=13)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost')
axes[0].set_yscale('log')

axes[1].plot(history_scaled, 'g-', linewidth=2)
axes[1].set_title('WITH Feature Scaling', fontsize=13)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Cost')
axes[1].set_yscale('log')

plt.suptitle('Impact of Feature Scaling on Convergence', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Key Insight: Feature scaling makes gradient descent converge MUCH faster!')
print(f'Unscaled final cost: {history_unscaled[-1]:.2f}')
print(f'Scaled final cost:   {history_scaled[-1]:.2f}')

### 5.3 Residual Analysis

In [ ]:
# ✅ SOLUTION: Residual analysis
# Fit model on full data for residual analysis
model_resid = LinearRegression()
model_resid.fit(X, y.ravel())
y_pred_resid = model_resid.predict(X)
residuals = y.ravel() - y_pred_resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Residuals vs Fitted
axes[0, 0].scatter(y_pred_resid, residuals, alpha=0.6, edgecolors='k', linewidth=0.3)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=1.5)
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted Values')

# (b) Histogram of Residuals
axes[0, 1].hist(residuals, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 1].set_xlabel('Residual Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Residuals')

# (c) Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normality Check)')

# (d) Residuals vs Feature
axes[1, 1].scatter(X, residuals, alpha=0.6, edgecolors='k', linewidth=0.3)
axes[1, 1].axhline(y=0, color='r', linestyle='--', linewidth=1.5)
axes[1, 1].set_xlabel('Feature X')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Residuals vs Feature Value')

plt.suptitle('Residual Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('What to look for:')
print('• Residuals vs Fitted: Random scatter → good. Pattern → assumption violated.')
print('• Histogram: Bell-shaped → normality assumption holds.')
print('• Q-Q Plot: Points on diagonal line → residuals are normal.')
print('• Residuals vs Feature: Random scatter → homoscedasticity holds.')

---
## ❓ Section 6: Interview Questions

See the practice notebook for all 7 interview questions with detailed answers.

---
## 🏆 Section 7: Challenge — Housing Price Prediction

In [ ]:
# ✅ SOLUTION: Challenge — Complete housing price prediction pipeline
np.random.seed(42)
n_houses = 500

sqft = np.random.uniform(500, 4000, n_houses)
bedrooms = np.random.randint(1, 6, n_houses).astype(float)
age = np.random.uniform(0, 50, n_houses)
distance = np.random.uniform(1, 30, n_houses)

price = (100 * sqft + 15000 * bedrooms - 2000 * age - 5000 * distance 
         + 50000 + np.random.randn(n_houses) * 20000)

X_housing = np.column_stack([sqft, bedrooms, age, distance])
y_housing = price.reshape(-1, 1)

print(f'Features shape: {X_housing.shape}')
print(f'Price range: ${y_housing.min():,.0f} — ${y_housing.max():,.0f}')

In [ ]:
from sklearn.preprocessing import StandardScaler

# Step 1: Train/test split
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

# Step 2: Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_h)
X_test_scaled = scaler.transform(X_test_h)

# Step 3: Add bias column for gradient descent
X_train_b = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
X_test_b = np.c_[np.ones((X_test_scaled.shape[0], 1)), X_test_scaled]

# Step 4: Gradient descent
theta_init = np.zeros((5, 1))  # 4 features + 1 bias
theta_housing, history_housing = gradient_descent(
    X_train_b, y_train_h, theta_init.copy(), alpha=0.1, num_iters=2000
)

# Predictions with gradient descent
y_pred_gd = X_test_b @ theta_housing

# Step 5: Also fit with sklearn
model_housing = LinearRegression()
model_housing.fit(X_train_h, y_train_h)
y_pred_sk = model_housing.predict(X_test_h)

# Step 6: Compare metrics
print('=== Gradient Descent ===')
print(f'  RMSE: ${np.sqrt(mean_squared_error(y_test_h, y_pred_gd)):,.0f}')
print(f'  R²:   {r2_score(y_test_h, y_pred_gd):.4f}')

print('\n=== scikit-learn ===')
print(f'  RMSE: ${np.sqrt(mean_squared_error(y_test_h, y_pred_sk)):,.0f}')
print(f'  R²:   {r2_score(y_test_h, y_pred_sk):.4f}')
print(f'\n  Intercept: {model_housing.intercept_[0]:,.0f}')
print(f'  Coefs: {model_housing.coef_[0]}')

In [ ]:
# Step 7: Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# GD results
axes[0].scatter(y_test_h, y_pred_gd, alpha=0.6, edgecolors='k', linewidth=0.3)
axes[0].plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], 
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Gradient Descent: Actual vs Predicted')
axes[0].legend()

# sklearn results
axes[1].scatter(y_test_h, y_pred_sk, alpha=0.6, edgecolors='k', linewidth=0.3, color='green')
axes[1].plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], 
             'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Price ($)')
axes[1].set_ylabel('Predicted Price ($)')
axes[1].set_title('scikit-learn: Actual vs Predicted')
axes[1].legend()

plt.suptitle('Housing Price Prediction', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Step 8: Residual analysis on sklearn model
residuals_h = y_test_h.ravel() - y_pred_sk.ravel()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_pred_sk, residuals_h, alpha=0.5, edgecolors='k', linewidth=0.3)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals_h, bins=25, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual Distribution')

stats.probplot(residuals_h, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

print('✅ Challenge complete!')

---
## ✅ Summary

Key takeaways:
- Linear regression finds θ that minimizes MSE
- Normal equation: fast for small n, O(n³)
- Gradient descent: iterative, works for any n, needs scaling
- Always check assumptions via residual analysis
- Know MSE vs RMSE vs MAE vs R² trade-offs for interviews

**Next**: [Polynomial Regression](../02-polynomial-regression/) →